In [2]:
import pandas as pd
import re

In [2]:
df = pd.read_json("News_Category_Dataset_v3.json",lines=True)
#df.head()

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [4]:
df.drop_duplicates(['headline','short_description'],keep='first',inplace=True)

In [5]:
df[df['headline'] == ''].value_counts().sum()
df = df[~(df['headline']=='')]

In [6]:
df[df['short_description'] == ''].value_counts().sum()
df = df[~(df['short_description']=='')]

In [7]:
df['full_news']  = df['headline'] + df['short_description']

df['full_news'].isnull().sum()

0

In [8]:
df['news_len'] = df['full_news'].apply(lambda x: len(x))

In [9]:
df['full_news'] = df['full_news'].str.lower()
df["full_news"] = df["full_news"].apply(lambda x: re.sub(r'[^a-zA-Z\s]', " ", x))
df['full_news'].tail()

209522    rim ceo thorsten heins   significant  plans fo...
209523    maria sharapova stunned by victoria azarenka i...
209524    giants over patriots  jets over colts among  m...
209525    aldon smith arrested    ers linebacker busted ...
209526    dwight howard rips teammates after magic loss ...
Name: full_news, dtype: object

In [10]:
import nltk
#nltk.download('all')

In [11]:
from nltk import word_tokenize

df['full_news'] = df['full_news'].apply(lambda x: nltk.word_tokenize(x))
df['full_news'].tail()

209522    [rim, ceo, thorsten, heins, significant, plans...
209523    [maria, sharapova, stunned, by, victoria, azar...
209524    [giants, over, patriots, jets, over, colts, am...
209525    [aldon, smith, arrested, ers, linebacker, bust...
209526    [dwight, howard, rips, teammates, after, magic...
Name: full_news, dtype: object

In [12]:
from nltk import FreqDist
from nltk.corpus import stopwords

In [13]:
stopwords = nltk.corpus.stopwords.words('english')
stopwords

['i',
 'me',
 'my',
 'myself',
 'we',
 'our',
 'ours',
 'ourselves',
 'you',
 "you're",
 "you've",
 "you'll",
 "you'd",
 'your',
 'yours',
 'yourself',
 'yourselves',
 'he',
 'him',
 'his',
 'himself',
 'she',
 "she's",
 'her',
 'hers',
 'herself',
 'it',
 "it's",
 'its',
 'itself',
 'they',
 'them',
 'their',
 'theirs',
 'themselves',
 'what',
 'which',
 'who',
 'whom',
 'this',
 'that',
 "that'll",
 'these',
 'those',
 'am',
 'is',
 'are',
 'was',
 'were',
 'be',
 'been',
 'being',
 'have',
 'has',
 'had',
 'having',
 'do',
 'does',
 'did',
 'doing',
 'a',
 'an',
 'the',
 'and',
 'but',
 'if',
 'or',
 'because',
 'as',
 'until',
 'while',
 'of',
 'at',
 'by',
 'for',
 'with',
 'about',
 'against',
 'between',
 'into',
 'through',
 'during',
 'before',
 'after',
 'above',
 'below',
 'to',
 'from',
 'up',
 'down',
 'in',
 'out',
 'on',
 'off',
 'over',
 'under',
 'again',
 'further',
 'then',
 'once',
 'here',
 'there',
 'when',
 'where',
 'why',
 'how',
 'all',
 'any',
 'both',
 'each

In [14]:
df['full_news'] = df['full_news'].apply(lambda x: [w for w in x if w.lower() not in stopwords] )
df['full_news']

0         [million, americans, roll, sleeves, omicron, t...
1         [american, airlines, flyer, charged, banned, l...
2         [funniest, tweets, cats, dogs, week, sept, dog...
3         [funniest, tweets, parents, week, sept, accide...
4         [woman, called, cops, black, bird, watcher, lo...
                                ...                        
209522    [rim, ceo, thorsten, heins, significant, plans...
209523    [maria, sharapova, stunned, victoria, azarenka...
209524    [giants, patriots, jets, colts, among, improba...
209525    [aldon, smith, arrested, ers, linebacker, bust...
209526    [dwight, howard, rips, teammates, magic, loss,...
Name: full_news, Length: 189426, dtype: object

In [15]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

sia.polarity_scores(df['short_description'][20000])

{'neg': 0.227, 'neu': 0.444, 'pos': 0.329, 'compound': 0.3182}

In [16]:
"""polarity_scores = []
for description in df['short_description']:
    polarity_scores.append(sia.polarity_scores(description))

# Convert the list of dictionaries to a DataFrame
polarity_df = pd.DataFrame(polarity_scores)

# Merge polarity DataFrame with the original DataFrame
df_with_polarity = pd.concat([df, polarity_df], axis=1)

# Display the DataFrame with polarity scores
print(df_with_polarity)"""

"polarity_scores = []\nfor description in df['short_description']:\n    polarity_scores.append(sia.polarity_scores(description))\n\n# Convert the list of dictionaries to a DataFrame\npolarity_df = pd.DataFrame(polarity_scores)\n\n# Merge polarity DataFrame with the original DataFrame\ndf_with_polarity = pd.concat([df, polarity_df], axis=1)\n\n# Display the DataFrame with polarity scores\nprint(df_with_polarity)"

In [17]:
sia.polarity_scores(df['short_description'][0])

{'neg': 0.051, 'neu': 0.949, 'pos': 0.0, 'compound': -0.128}

In [18]:
# Define a function to apply SIA on tokenized text
def sia_on_tokenized(tokenized_text):
    # Initialize variables to store sentiment scores
    total_neg = 0
    total_neu = 0
    total_pos = 0
    total_compound = 0

    # Apply SIA to each token
    for token in tokenized_text:
        scores = sia.polarity_scores(token)
        total_neg += scores['neg']
        total_neu += scores['neu']
        total_pos += scores['pos']
        total_compound += scores['compound']

    # Calculate average sentiment scores
    num_tokens = len(tokenized_text)
    avg_neg = round(total_neg / num_tokens, 3)
    avg_neu = round(total_neu / num_tokens, 3)
    avg_pos = round(total_pos / num_tokens, 3)
    avg_compound = round(total_compound / num_tokens, 3)


    #label based on compound 
    compound_score = scores['compound']
    
    # Label sentiment based on compound score
    if compound_score > 0.5:
        return 'positive'
    elif compound_score < -0.5:
        return 'negative'
    else:
        return 'neutral'

    #return {
        #'neg': avg_neg,
        #'neu': avg_neu,
        #'pos': avg_pos,
        #'compound': avg_compound
    #}


In [19]:
#sia on full news and label baseed on compound
df['sentiment'] = df['full_news'].apply(lambda x: sia_on_tokenized(x))
df['sentiment'].head()

0    neutral
1    neutral
2    neutral
3    neutral
4    neutral
Name: sentiment, dtype: object

In [20]:
df['sentiment'].value_counts()

sentiment
neutral     181923
positive      3898
negative      3605
Name: count, dtype: int64

In [24]:
df.to_excel("C:/Users/Aditi/OneDrive/Desktop/Data Analysis/Internship/inshorts/sia.xlsx")

start here

In [3]:
# sia.xlsx

df = pd.read_excel("C:/Users/Aditi/OneDrive/Desktop/Data Analysis/Internship/inshorts/sia.xlsx")

In [71]:
df.drop(columns=['Unnamed: 0','link','authors','date','news_len'],inplace=True)

In [72]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189426 entries, 0 to 189425
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   headline           189426 non-null  object
 1   category           189426 non-null  object
 2   short_description  189426 non-null  object
 3   full_news          189426 non-null  object
 4   sentiment          189426 non-null  object
dtypes: object(5)
memory usage: 7.2+ MB


,headline,category,short_description,full_news,sentiment
count,189426,189426,189426,189426,189426
unique,188417,42,187020,189168,3
top,Sunday Roundup,POLITICS,Welcome to the HuffPost Rise Morning Newsbrief...,"['time', 'tip', 'day', 'september', 'need', 't...",neutral
freq,90,32427,192,21,181923


In [ ]:
from sklearn.model_selection  import train_test_split
from sklearn.metrics import accuracy_score 
import pandas as pd

In [74]:

positive_df = df[df['sentiment'] == 'positive'].head(3500)
neutral_df = df[df['sentiment'] == 'neutral'].head(3500)
negative_df = df[df['sentiment'] == 'negative'].head(3500)

# Concatenate 
final_df = pd.concat([positive_df, neutral_df, negative_df])

# Shuffle 
final_df = final_df.sample(frac=1).reset_index(drop=True)


In [75]:
final_df.head()

,headline,category,short_description,full_news,sentiment
0,Our Money Addiction Problem,WELLNESS,We have a serious money addiction problem in o...,"['money', 'addiction', 'problemwe', 'serious',...",positive
1,Father's Day Art: Best Portrayals Of Fatherhoo...,CULTURE & ARTS,"On this momentous occasion of Father's Day, we...","['father', 'day', 'art', 'best', 'portrayals',...",positive
2,U.S. Allies Threaten Retaliation Over Trump's ...,WORLD NEWS,"The president, who announced the protectionist...","['u', 'allies', 'threaten', 'retaliation', 'tr...",positive
3,Paul Ryan's War On Social Security,POLITICS,Mr. Ryan has made quite a career and a name fo...,"['paul', 'ryan', 'war', 'social', 'securitymr'...",negative
4,How to Celebrate Halloween Like Grown-up,FOOD & DRINK,"How to host a ""Halloween Dinner Party"" that's ...","['celebrate', 'halloween', 'like', 'grown', 'u...",positive


In [76]:
y = final_df['sentiment']
final_df.drop(columns=['sentiment'],inplace=True)

In [108]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import   StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import make_pipeline,Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [102]:
X_train,x_test,y_train,y_test = train_test_split(final_df['full_news'],y,random_state=42,shuffle=True)

In [101]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [103]:
vectorizer = CountVectorizer()

# Create a pipeline with CountVectorizer and LogisticRegression
pipeline = make_pipeline(vectorizer, LogisticRegression())

# Train the model
pipeline.fit(X_train, y_train)

# Predict on the test set
y_pred = pipeline.predict(x_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{report}")
print(f"Confusion Matrix:\n{conf_matrix}")

Pipeline(steps=[('countvectorizer', CountVectorizer()),
                ('logisticregression', LogisticRegression())])

In [ ]:
log = LogisticRegression()
log.fit(X_train_cv, y_train)
y_pred = log.predict(X_test_cv)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{report}")
print(f"Confusion Matrix:\n{conf_matrix}")

Accuracy: 0.8045714285714286
Classification Report:
              precision    recall  f1-score   support

    negative       0.77      0.78      0.78       841
     neutral       0.80      0.78      0.79       871
    positive       0.84      0.85      0.84       913

    accuracy                           0.80      2625
   macro avg       0.80      0.80      0.80      2625
weighted avg       0.80      0.80      0.80      2625

Confusion Matrix:
[[657 103  81]
 [120 681  70]
 [ 77  62 774]]


In [127]:
from sklearn.naive_bayes import (
    BernoulliNB,
    ComplementNB,
    MultinomialNB,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.svm import SVC

In [ ]:
# Create a pipeline with the classifiers
pipeline2 = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    (#'multi', MultinomialNB(),
     #'comple',ComplementNB(),
     #'bern',BernoulliNB(),
     'log',LogisticRegression(),
     #'kneigh',KNeighborsClassifier(),
     #'dtree',DecisionTreeClassifier(),
     'rf_clf',RandomForestClassifier(),
     'svm',SVC(),
     #'ada',AdaBoostClassifier()
     #'mlp',MLPClassifier(),
     ),
])
# Predict on the test set
pipeline2.fit(X_train, y_train)
y_pred = pipeline2.predict(x_test)


# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{report}")
print(f"Confusion Matrix:\n{conf_matrix}")

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42),#maximum likelihood  
    #'KNN':KNeighborsClassifier(n_neighbors=3),
    #'Poly_SVC': SVC(kernel="poly", degree=3, coef0=1, C=5),
    #'sgd': SGDClassifier(),
    #'rf': RandomForestClassifier(),
}

# Iterate over classifiers and perform cross-validation
for clf_name, clf in classifiers.items():
    pipeline = Pipeline([
        ('classifier', clf)
    ])
    # Perform cross-validation on the training set
    scores = cross_val_score(pipeline, X_train_cv, y_train, cv=5)
    print(f'{clf_name} Cross-Validation Accuracy: {scores.mean()}')

    # Fitting the classifier to the training data
    pipeline.fit(X_train_cv, y_train)

    # Predict on train and test set
    y_train_pred = pipeline.predict(X_train_cv)
    y_pred = pipeline.predict(X_test_cv)

    
    # Performance and confusion  matrix
    print(classification_report(y_train, y_train_pred)) 
    cm_train = confusion_matrix(y_train,y_train_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_train)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'{clf_name} Train')
    plt.show()

    print(classification_report(y_test,y_pred))
    cm_test = confusion_matrix(y_test,y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_test)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'{clf_name} Test')
    plt.show()


    #save models
    # models created
    #model_filename = f'{clf_name.replace(" ", "_")}_model.pkl'
    #with open(model_filename, 'wb') as file:
        #pickle.dump(clf,file)


#load model
#model_filename =path/to/model
# pickled_model = pickle.load(open(model_filename, 'rb'))
#pickled_model.predict(x_test)


In [9]:
data1.columns

Index(['headline', 'category', 'short_description', 'authors', 'date',
       'source_website', 'token_desc', 'token_headline', 'sentiment', 'tags',
       'Pos_headline', 'Pos', 'lemm', 'stem', 'lemm_headline',
       'stem_headline'],
      dtype='object')

In [24]:
data1['full_news'] = (data2['full_news'])

In [22]:
data1['sentiment_full'] = (data2['sentiment'])

In [25]:
data1.columns

Index(['headline', 'category', 'short_description', 'authors', 'date',
       'source_website', 'token_desc', 'token_headline', 'sentiment', 'tags',
       'Pos_headline', 'Pos', 'lemm', 'stem', 'lemm_headline', 'stem_headline',
       'sentiment_full', 'full_news'],
      dtype='object')

In [26]:
data1.to_pickle("data_final.pkl")

In [27]:
data3 = pd.read_pickle("data_final.pkl")

In [28]:
data3.head()

,headline,category,short_description,authors,date,source_website,token_desc,token_headline,sentiment,tags,Pos_headline,Pos,lemm,stem,lemm_headline,stem_headline,sentiment_full,full_news
0,over million americans roll up sleeves for o...,WORLD NEWS,health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23,huffpost,"[health, experts, said, early, predict, whethe...","[million, americans, roll, sleeves, omicron, t...",-1,"[(million, CARDINAL)]","[(million, CD), (americans, NNS), (roll, VBP),...","[(health, NN), (experts, NNS), (said, VBD), (e...","[health, expert, said, early, predict, whether...","[health, expert, said, earli, predict, whether...","[million, american, roll, sleeve, omicron, tar...","[million, american, roll, sleev, omicron, targ...",neutral,"['million', 'americans', 'roll', 'sleeves', 'o..."
1,american airlines flyer charged banned for li...,WORLD NEWS,he was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23,huffpost,"[subdued, passengers, crew, fled, back, aircra...","[american, airlines, flyer, charged, banned, l...",-1,"[(angeles, GPE)]","[(american, JJ), (airlines, NNS), (flyer, RB),...","[(subdued, VBN), (passengers, NNS), (crew, VBD...","[subdued, passenger, crew, fled, back, aircraf...","[subdu, passeng, crew, fled, back, aircraft, c...","[american, airline, flyer, charged, banned, li...","[american, airlin, flyer, charg, ban, life, pu...",neutral,"['american', 'airlines', 'flyer', 'charged', '..."
2,of the funniest tweets about cats and dogs ...,ENTERTAINMENT,until you have a dog you don t understand wha...,Elyse Wanshel,2022-09-23,huffpost,"[dog, understand, could, eaten]","[funniest, tweets, cats, dogs, week, sept]",0,[],"[(funniest, JJS), (tweets, NNS), (cats, NNS), ...","[(dog, NN), (understand, NN), (could, MD), (ea...","[dog, understand, could, eaten]","[dog, understand, could, eaten]","[funniest, tweet, cat, dog, week, sept]","[funniest, tweet, cat, dog, week, sept]",neutral,"['funniest', 'tweets', 'cats', 'dogs', 'week',..."
3,the funniest tweets from parents this week se...,RELATIONSHIPS,accidentally put grown up toothpaste on my to...,Caroline Bologna,2022-09-23,huffpost,"[accidentally, put, grown, toothpaste, toddler...","[funniest, tweets, parents, week, sept]",-1,"[(carolina, GPE), (tabasco, ORG)]","[(funniest, JJS), (tweets, NNS), (parents, NNS...","[(accidentally, RB), (put, VBD), (grown, JJ), ...","[accidentally, put, grown, toothpaste, toddler...","[accident, put, grown, toothpast, toddler, too...","[funniest, tweet, parent, week, sept]","[funniest, tweet, parent, week, sept]",neutral,"['funniest', 'tweets', 'parents', 'week', 'sep..."
4,woman who called cops on black bird watcher lo...,WORLD NEWS,amy cooper accused investment firm franklin te...,Nina Golgowski,2022-09-22,huffpost,"[amy, cooper, accused, investment, firm, frank...","[woman, called, cops, black, bird, watcher, lo...",-1,"[(amy, PERSON), (cooper, ORG), (franklin, ORG)...","[(woman, NN), (called, VBN), (cops, NNS), (bla...","[(amy, JJ), (cooper, NN), (accused, VBN), (inv...","[amy, cooper, accused, investment, firm, frank...","[ami, cooper, accus, invest, firm, franklin, t...","[woman, called, cop, black, bird, watcher, los...","[woman, call, cop, black, bird, watcher, lose,...",neutral,"['woman', 'called', 'cops', 'black', 'bird', '..."
